In [1]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [2]:
df = pd.read_csv(r'/home/user/Downloads/DataHut_CH_Lidl_PriceExtractions_20260814.CSV', low_memory=False, sep='|')

In [3]:
mask = (
    df["product_unique_key"].astype(str)
    !=
    (df["unique_id"].astype(str) + "P")
)

issues = df.loc[mask, ["unique_id", "product_unique_key", "pdp_url"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "unique_id", "product_unique_key"]]

issues

KeyError: 'product_unique_key'

In [3]:
df.isnull().sum()

unique_id                   0
competitor_name             0
extraction_date             0
product_name                0
grammage_quantity           0
grammage_unit               0
regular_price               0
selling_price               0
price_valid_from         3044
price_per_unit            432
percentage_discount      3315
promotion_price          3313
promotion_valid_from     3044
promotion_valid_upto     3366
promotion_description    2980
currency                    0
breadcrumb                  0
pdp_url                     0
region                   3620
pack_size                3620
site_shown_uom            432
dtype: int64

In [4]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

issues = {}

for col in text_cols:
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(r"(?<!\\)\|", regex=True)
    )

    if mask.any():
        temp = df.loc[mask, ["pdp_url", col]].copy()
        temp.insert(0, "row_no", temp.index)  # DataFrame row number
        issues[col] = temp

# Print results
if issues:
    print("Columns containing unescaped '|':\n")

    for col, data in issues.items():
        print(f"\n=== {col} ({len(data)} rows) ===")
        print(data.to_string(index=False))
else:
    print("No unescaped '|' found.")

No unescaped '|' found.


In [5]:
import pandas as pd

escape_pattern = r'[\n\r\t\b\f\v]'

issues = []

for col in df.select_dtypes(include="object").columns:
    mask = df[col].fillna("").astype(str).str.contains(
        escape_pattern,
        regex=True,
        na=False
    )

    if mask.any():
        temp = df.loc[mask, [col]].copy()
        temp.insert(0, "row_no", temp.index + 2)
        temp["column"] = col
        temp["issue_value"] = temp[col]

        if "pdp_url" in df.columns:
            temp["pdp_url"] = df.loc[mask, "pdp_url"]

        issues.append(
            temp[["row_no", "pdp_url", "column", "issue_value"]]
        )

if issues:
    result = pd.concat(issues, ignore_index=True)

    print("=== Escape Character Issues ===")
    print(result.to_string(index=False))
    print(f"\nTotal issues: {len(result)}")
else:
    print("No escape characters found.")

No escape characters found.


In [5]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

issues = {}

for col in text_cols:
    # Match | not preceded by a backslash
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(r"(?<!\\)\|", regex=True)
    )

    if mask.any():
        issues[col] = df.loc[mask, ["unique_id", col]]

# Print results
if issues:
    print("Columns containing unescaped '|':\n")
    for col, data in issues.items():
        print(f"\n=== {col} ({len(data)} rows) ===")
        print(data.to_string(index=False))
else:
    print("No unescaped '|' found.")

No unescaped '|' found.


In [7]:
df.shape

(3620, 21)

In [13]:
empty_cols = df.columns[df.isna().all()].tolist()
empty_cols

['price_valid_from',
 'percentage_discount',
 'promotion_valid_from',
 'promotion_valid_upto',
 'region',
 'pack_size']

In [8]:
req_cols = """unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,
site_shown_uom"""

req_cols = [c.strip() for c in req_cols.split(",") if c.strip()]

missing = [c for c in req_cols if c not in df.columns]
extra = [c for c in df.columns if c not in req_cols]

print("Requirement columns" \
":", len(req_cols))
print("File columns:", len(df.columns))
print("Missing:", missing)
print("Extra:", extra)
print("Order matches:", req_cols == list(df.columns))

Requirement columns: 21
File columns: 21
Missing: []
Extra: []
Order matches: True


In [9]:
df.competitor_name.value_counts()

competitor_name
lidl    3620
Name: count, dtype: int64

In [10]:
# Empty columns as per requirement
req_empty = """ """

req_empty = [c.strip() for c in req_empty.split(",") if c.strip()]

# Empty columns in the file
file_empty = df.columns[df.isna().all()].tolist()

# Compare
missing_empty = [c for c in req_empty if c not in file_empty]
unexpected_empty = [c for c in file_empty if c not in req_empty]
matching_empty = [c for c in req_empty if c in file_empty]

print("Requirement empty columns :", len(req_empty))
print("File empty columns        :", len(file_empty))
print("Matching                 :", len(matching_empty))
print("Expected empty but not empty:", missing_empty)
print("Unexpected empty columns     :", unexpected_empty)

Requirement empty columns : 0
File empty columns        : 2
Matching                 : 0
Expected empty but not empty: []
Unexpected empty columns     : ['region', 'pack_size']


In [12]:
df.isnull().sum()

unique_id                    0
competitor_name              0
extraction_date              0
product_name                 0
grammage_quantity            0
grammage_unit                0
regular_price               19
selling_price               19
price_valid_from         41191
price_per_unit           16995
percentage_discount      41191
promotion_price          38938
promotion_valid_from     41191
promotion_valid_upto     41191
promotion_description    38938
currency                     0
breadcrumb                   0
pdp_url                      0
region                   41191
pack_size                41191
site_shown_uom               0
dtype: int64

In [11]:
df.nunique()

unique_id                3620
competitor_name             1
extraction_date             1
product_name             3207
grammage_quantity         396
grammage_unit               5
regular_price             348
selling_price             378
price_valid_from            7
price_per_unit           1012
percentage_discount        32
promotion_price           122
promotion_valid_from        7
promotion_valid_upto        6
promotion_description      67
currency                    1
breadcrumb               3005
pdp_url                  3620
region                      0
pack_size                   0
site_shown_uom            716
dtype: int64

In [19]:
df.competitor_product_key.value_counts()

AttributeError: 'DataFrame' object has no attribute 'competitor_product_key'

In [12]:
df.unique_id.duplicated().sum()

np.int64(0)

In [13]:
df.instock.value_counts()

AttributeError: 'DataFrame' object has no attribute 'instock'

In [14]:
df.grammage_unit.value_counts()

grammage_unit
g        2047
stück     714
l         398
ml        352
kg        109
Name: count, dtype: int64

In [15]:
df.competitor_name.value_counts()

competitor_name
lidl    3620
Name: count, dtype: int64

In [16]:
pattern = r'^[^>]+( > [^>]+)*$'

# Invalid breadcrumbs
invalid = df[
    df["breadcrumb"].isna() |
    (df["breadcrumb"].str.strip() == "") |
    (~df["breadcrumb"].str.fullmatch(pattern, na=False))
]

print(f"Total invalid breadcrumbs: {len(invalid)}")

if not invalid.empty:
    print(invalid[["breadcrumb"]])

Total invalid breadcrumbs: 0


In [17]:
df.currency.value_counts()

currency
Swiss franc    3620
Name: count, dtype: int64

In [18]:
import pandas as pd
import re

price_columns = [
    "grammage_quantity",
    "regular_price",
    "selling_price",
    "price_was",
    "promotion_price",
    "percentage_discount",
    "price_per_unit",
    "multi_buy_items_price_total",
    "multibuy_items_pricesingle"
]

for col in price_columns:

    if col not in df.columns:
        continue

    s = df[col].fillna("").astype(str).str.strip()

    # Extract only numeric-looking part
    # This allows values such as:
    # KG. za 3.58 €
    # KG. za 0.9 €
    number = s.str.extract(r"(\d[\d.,]*)", expand=False).fillna("")

    # --------------------------------------------------
    # 1. MULTIPLE DECIMAL POINTS
    # --------------------------------------------------
    multiple_decimal = number.str.count(r"\.") > 1

    # --------------------------------------------------
    # 2. COMMA USED INSTEAD OF DECIMAL POINT
    # --------------------------------------------------
    comma_decimal = number.str.contains(
        r",",
        regex=True,
        na=False
    )

    # --------------------------------------------------
    # 3. EXACTLY 2 DECIMAL PLACES
    # ONLY regular_price and selling_price
    # --------------------------------------------------
    two_decimal_issue = pd.Series(False, index=df.index)

    if col in ["regular_price", "selling_price"]:
        two_decimal_issue = (
            (s != "") &
            (~s.str.fullmatch(r"\d+\.\d{2}", na=False))
        )

    print(f"\n{'='*15} {col} {'='*15}")

    if multiple_decimal.any():
        print(f"Multiple decimal points: {multiple_decimal.sum()}")
        print(
            df.loc[multiple_decimal, [col]]
            .head(10)
            .to_string(index=False)
        )

    if comma_decimal.any():
        print(f"Comma used instead of dot: {comma_decimal.sum()}")
        print(
            df.loc[comma_decimal, [col]]
            .head(10)
            .to_string(index=False)
        )

    if two_decimal_issue.any():
        print(f"Not in 2-decimal format: {two_decimal_issue.sum()}")
        print(
            df.loc[two_decimal_issue, [col]]
            .head(10)
            .to_string(index=False)
        )


=============== grammage_quantity ===============

=============== regular_price ===============
Not in 2-decimal format: 58
 regular_price
           2.6
           3.3
           8.5
          16.0
          10.0
          24.0
           2.7
          26.5
           8.5
           0.1

=============== selling_price ===============
Not in 2-decimal format: 61
 selling_price
           2.6
           3.3
          16.0
          10.0
          24.0
           2.7
           3.0
           2.2
           0.1
           7.8

=============== promotion_price ===============

=============== percentage_discount ===============

=============== price_per_unit ===============


In [19]:
import pandas as pd

price_columns = [
    "grammage_quantity",
    "regular_price",
    "selling_price",
    "price_was",
    "promotion_price",
    "percentage_discount",
    "price_per_unit",
    "multi_buy_items_price_total",
    "multibuy_items_pricesingle"
]

for col in price_columns:
    if col not in df.columns:
        continue

    s = df[col].fillna("").astype(str)

    issues = {
        "Zero values": pd.to_numeric(s, errors="coerce") == 0,
        "Negative values": pd.to_numeric(s, errors="coerce") < 0,
        "Multiple decimal points": s.str.count(r"\.") > 1,
        "More than 2 decimal places": s.str.contains(r"\.\d{3,}$", regex=True),
        "Non-numeric values": (~s.str.match(r"^-?\d+(\.\d+)?$", na=False)) & (s != ""),
        "Leading/Trailing whitespace": s != s.str.strip(),
    }

    print(f"\n{'='*15} {col} {'='*15}")

    for issue, mask in issues.items():
        if mask.any():
            print(f"{issue}: {mask.sum()}")
            print(df.loc[mask, [col]].head(5).to_string(index=False))


=============== grammage_quantity ===============
More than 2 decimal places: 8
grammage_quantity
          6x0.275
            1.491
            1.491
            1.125
            1.125
Non-numeric values: 314
grammage_quantity
            6x0.5
            12x60
            8x100
            4x0.5
            4x100

=============== regular_price ===============

=============== selling_price ===============

=============== promotion_price ===============

=============== percentage_discount ===============

=============== price_per_unit ===============
Non-numeric values: 3188
 price_per_unit
    pro 1 stück
    pro 1 stück
       pro 1 kg
 1Kg = 3.50 CHF
100g = 1.50 CHF


In [20]:
mask = (
    df["price_valid_from"].notna() &
    df["promotion_valid_from"].notna() &
    (df["price_valid_from"] != df["promotion_valid_from"])
)

issues = df.loc[
    mask,
    ["pdp_url", "price_valid_from", "promotion_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues[["row_no", "pdp_url", "price_valid_from", "promotion_valid_from"]]

,row_no,pdp_url,price_valid_from,promotion_valid_from


In [21]:
import re

col = "regular_price"

# More than one decimal point in the numeric part
invalid = df[
    df[col]
    .fillna("")
    .astype(str)
    .str.contains(r'\d+\.\d+\.', regex=True)
]

print(f"Rows with multiple decimal points: {len(invalid)}")
print(invalid[[col]])

Rows with multiple decimal points: 0
Empty DataFrame
Columns: [regular_price]
Index: []


In [22]:
import re

url_pattern = re.compile(
    r"^https?://[^\s/$.?#].[^\s]*$",
    re.IGNORECASE
)

invalid_urls = df[
    
    df["pdp_url"].isna() |
    ~df["pdp_url"].astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_urls)}")
print(invalid_urls[[ "pdp_url"]])

Invalid URLs: 0
Empty DataFrame
Columns: [pdp_url]
Index: []


In [23]:
df.loc[df["pdp_url"].str.contains(r"\s", regex=True, na=False),
       ["pdp_url"]]

,pdp_url


In [24]:
import pandas as pd

# Promotion description has a value containing %
promo_mask = (
    df["promotion_description"].fillna("").astype(str).str.contains("%", na=False)
)

# Percentage discount is empty
discount_empty = (
    df["percentage_discount"].isna() |
    df["percentage_discount"].astype(str).str.strip().eq("")
)

# Both conditions
mask = promo_mask & discount_empty

issues = df.loc[
    mask,
    ["promotion_description", "percentage_discount", "pdp_url"]
].copy()

# Row number (assuming header is row 1)
issues["row_no"] = issues.index + 2

# Arrange columns
issues = issues[
    ["row_no", "pdp_url", "promotion_description", "percentage_discount"]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, percentage_discount]
Index: []


In [27]:
df.grammage_quantity.value_counts()

grammage_quantity
1           14874
0.75         2233
500          1927
100          1878
250          1621
200          1341
150           953
0.5           928
400           856
0.7           809
20            677
300           650
330           575
50            571
350           398
125           364
75            360
180           296
80            288
10            274
1.5           248
40            236
60            235
450           225
70            219
2             205
30            203
120           198
3             194
600           180
90            175
750           169
25            164
220           145
240           143
4             140
190           131
160           129
175           128
375           126
140           125
360           115
800           112
110           109
5             101
6             100
16             99
130            96
8              95
340            92
35             91
85             88
18             88
225            87
45        

In [25]:
mask = (
    df["promotion_valid_from"].notna()
    & df["promotion_valid_from"].astype(str).str.strip().ne("")
    & (
        df["price_valid_from"].isna()
        | df["price_valid_from"].astype(str).str.strip().eq("")
    )
)

issues = df.loc[
    mask,
    
    [ "pdp_url", "promotion_valid_from", "price_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no",  "pdp_url", "promotion_valid_from", "price_valid_from"]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, pdp_url, promotion_valid_from, price_valid_from]
Index: []


In [26]:
invalid_urls = invalid_urls["pdp_url"].unique()

print("Unique invalid URLs:", len(invalid_urls))
for url in invalid_urls:
    print(repr(url))

Unique invalid URLs: 0


In [31]:
import numpy as np

invalid_urls = np.unique(invalid_urls)

print("Unique invalid URLs:", len(invalid_urls))
for url in invalid_urls:
    print(url)  # Or your specific loop logic


Unique invalid URLs: 0


In [32]:
issues = df[
    df["product_name"].astype(str).str.startswith("Denner", na=False)
    & (df["brand"].fillna("").str.strip() != "Denner")
]

issues[["product_name", "brand", "pdp_url"]]

KeyError: 'brand'

In [33]:
df.instock.value_counts()

AttributeError: 'DataFrame' object has no attribute 'instock'

In [27]:
import re

image_cols = [c for c in df.columns if c.startswith("image_url_")]

# Basic image URL pattern
pattern = re.compile(
    r"^https?://[^\s]+\.(jpg|jpeg|png|webp|gif|bmp|avif)(\?.*)?$",
    re.IGNORECASE
)

for col in image_cols:
    invalid = df[
        df[col].notna() &
        (df[col].astype(str).str.strip() != "") &
        ~df[col].astype(str).str.match(pattern)
    ]

    print(f"\n{col}: {len(invalid)} invalid URLs")

    if not invalid.empty:
        print(invalid[["unique_id", col]].head())

In [28]:
(
    df.groupby(['store_addressline2', 'unique_id'])
      .size()
      .reset_index(name='count')
      .query('count > 1')
      .sort_values(['store_addressline2', 'count'], ascending=[True, False])
)

KeyError: 'store_addressline2'

In [29]:
df.extraction_date.value_counts()

extraction_date
2026-08-14    3620
Name: count, dtype: int64

In [30]:
import re

pattern = re.compile(r"^https?://[^\s]+$", re.IGNORECASE)

for col in image_cols:
    invalid = df[
        df[col].notna() &
        (df[col].astype(str).str.strip() != "") &
        ~df[col].astype(str).str.match(pattern)
    ]

    print(f"{col}: {len(invalid)} invalid URLs")

In [31]:
price_cols = [
    "regular_price",
    "selling_price",
   
   
   
]

# Keep only columns that exist in the file
price_cols = [c for c in price_cols if c in df.columns]

# Rows where at least one price column is empty
empty_price_rows = df[df[price_cols].replace("", pd.NA).isna().any(axis=1)]

print(f"Rows with empty price fields: {len(empty_price_rows)}")
display(empty_price_rows[["unique_id","pdp_url", "promotion_description","product_name"] + price_cols])

Rows with empty price fields: 0


,unique_id,pdp_url,promotion_description,product_name,regular_price,selling_price


In [32]:
import pandas as pd

# Treat empty strings as missing values
promo_price = df["promotion_price"].replace("", pd.NA)
promo_desc = df["promotion_description"].replace("", pd.NA)

# 1. Promotion price present but promotion description missing
price_no_desc = df[
    promo_price.notna() & promo_desc.isna()
]

# 2. Promotion description present but promotion price missing
desc_no_price = df[
    promo_desc.notna() & promo_price.isna()
]

print("Promotion price present but description missing:", len(price_no_desc))
display(price_no_desc[["unique_id", "product_name", "promotion_price","pdp_url", "promotion_description"]])

print("\nPromotion description present but price missing:", len(desc_no_price))
display(desc_no_price[["unique_id", "product_name", "promotion_price", "pdp_url","promotion_description"]])

Promotion price present but description missing: 0


,unique_id,product_name,promotion_price,pdp_url,promotion_description



Promotion description present but price missing: 333


,unique_id,product_name,promotion_price,pdp_url,promotion_description
34,5114255,Chicken Wings nature,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21941/s/chicken-wings-nature-5114255/category/54/,Aktion
35,1801686,Frisco Extrême Cornets Soft Core,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21940/s/frisco-extreme-cornets-soft-core-1801686/category/54/,Aktion
38,1009870,Glace Erdbeer Crisp XXL,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21939/s/glace-erdbeer-crisp-xxl-1009870/category/54/,Aktion
42,1021254,Lager Bier XXL,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21938/s/lager-bier-xxl-1021254/category/54/,Aktion
43,1017455,Babyfeuchttücher 99% Wasser XXL,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21937/s/babyfeuchttuecher-99-wasser-xxl-1017455/category/54/,Aktion
44,153647,Mini Mix Mandel Vollmilch,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21936/s/mini-mix-mandel-vollmilch-0153647/category/54/,Aktion
45,218173,Hummus XXL,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21934/s/hummus-xxl-0218173/category/54/,Aktion
46,215443,Skyr Natur 0.2% XXL,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21932/s/skyr-natur-0-2-xxl-0215443/category/54/,Aktion
47,1018612,ASC XXL Riesencrevetten,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21931/s/asc-xxl-riesencrevetten-1018612/category/54/,Aktion
48,5719125,Blütenhonig XXL,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21935/s/bluetenhonig-xxl-5719125/category/54/,Aktion


In [40]:
df.breadcrumb.value_counts()

breadcrumb
SPAR Website > Produktwelt > Haushalt > Textilien > Wäsche & Strumpfwaren > Wäsche Damen                                                              1381
SPAR Website > Produktwelt > Haushalt > Freizeit & Saisonen > Dekoration > Saisonale Dekoration                                                       1028
SPAR Website > Produktwelt > Getränke > Alkoholische Getränke > Weine > Weißweine                                                                      885
SPAR Website > Produktwelt > Getränke > Alkoholische Getränke > Weine > Rotweine                                                                       827
SPAR Website > Produktwelt > Haushalt > Textilien > Wäsche & Strumpfwaren > Strumpfwaren                                                               768
SPAR Website > Produktwelt > Haushalt > Freizeit & Saisonen > Dekoration > Kerzen & Raumduft                                                           742
SPAR Website > Produktwelt > Lebensmittel > Beilagen, Essig

In [33]:
invalid = df[df["breadcrumb"].str.contains("…", na=False)]

print(f"Rows with truncated breadcrumbs: {len(invalid)}")
display(invalid[["unique_id", "breadcrumb"]])

Rows with truncated breadcrumbs: 0


,unique_id,breadcrumb


In [34]:
result = df[
    df["selling_price"].isna() & df["regular_price"].isna()

text_cols = df.select_dtypes(include="object").columns

issues = {}

for col in text_cols:
    # Match | not preceded by a backslash
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(r"(?<!\\)\|", regex=True)
    )

    if mask.any():
        issues[col] = df.loc[mask, ["unique_id", col]]

# Print results
if issues:
    print("Columns containing unescaped '|':\n")
    for col, data in issues.items():
        print(f"\n=== {col} ({len(data)} rows) ===")
        print(data.to_string(index=False))
else:
    print("No unescaped '|' found.")
]

print(result)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3466212421.py, line 2)

In [35]:
import pandas as pd

# Text columns only
text_cols = df.select_dtypes(include="object").columns

invalid_chars = [ "\n", "\r", "\t"]

issues = {}

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r"[\n\r\t]", regex=True)
    if mask.any():
        issues[col] = df.loc[mask, ["unique_id", col]]

# Summary
if issues:
    print("Columns containing \\n, \\r or \\t:\n")
    for col, data in issues.items():
        print(f"{col}: {len(data)} rows")
else:
    print("No \\n, \\r or \\t found.")
    

No \n, \r or \t found.


In [36]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

results = []

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r"[\n\r\t]", regex=True)

    for idx, value in df.loc[mask, col].items():
        results.append({
            "row_no": idx + 2,      # Excel row number
            "unique_id": df.at[idx, "unique_id"],
            "pdp_url": df.at[idx, "pdp_url"],
            "column": col,
            "issue_value": value
        })

issues_df = pd.DataFrame(results)

issues_df

""


In [37]:
columns = ["product_description", "special_information", "manufacturer_address"]

for col in columns:
    rows = df[
        df[col]
        .fillna("")
        .astype(str)
        .str.contains("|", regex=False, na=False)
    ]

    print(f"\n=== {col}: {len(rows)} rows ===")

    if not rows.empty:
        display(rows[["unique_id", "pdp_url", col]])

KeyError: 'product_description'

In [46]:
import pandas as pd
import re




df["dimensions"] = df["dimensions"].fillna("").str.strip()

pattern = re.compile(
    r'^\d+(\.\d+)?(\s*[a-zA-Z]+)?\s*[xX×]\s*'
    r'\d+(\.\d+)?(\s*[a-zA-Z]+)?\s*[xX×]\s*'
    r'\d+(\.\d+)?(\s*[a-zA-Z]+)?$'
)

invalid = df[
    (df["dimensions"] != "") &
    (~df["dimensions"].str.match(pattern))
]

print(f"Invalid values: {len(invalid)}")
print(invalid[["unique_id", "product_name", "dimensions"]])

KeyError: 'dimensions'

In [38]:
import pandas as pd

issues = []

for col in df.columns:
    if df[col].dtype == "object":
        mask = df[col].fillna("").astype(str).str.contains(
            r'^\s+|\s+$|\t|\n|\r| {2,}',
            regex=True,
            na=False
        )

        if mask.any():
            temp = df.loc[mask, [col]].copy()
            temp["row_no"] = temp.index + 2
            temp["column"] = col
            issues.append(temp[["row_no", "column", col]])

if issues:
    result = pd.concat(issues, ignore_index=True)
    print(result)
else:
    print("No unwanted whitespace found.")

   row_no        column                               product_name  \
0    3539  product_name  Mini Pizza Pancetta, Provola und  Zwiebel   
1    3539    breadcrumb                                        NaN   

                                                          breadcrumb  
0                                                                NaN  
1  Sortiment > Sortiment > Mini Pizza Pancetta, Provola und  Zwiebel  


In [48]:

# Replace empty strings with NA
df = df.replace(r'^\s*$', pd.NA, regex=True)

# Filter Fleisch & Fisch
ff = df[df["producthierarchy_level1"] == "Fleisch & Fisch"]

# Subcategories that should have price_per_unit
ppu_subcats = ["Fleisch & Geflügel", "Frischfisch & Schalentiere"]

# Missing price_per_unit
invalid_ppu = ff[
    ff["producthierarchy_level2"].isin(ppu_subcats) &
    ff["price_per_unit"].isna()
]

print("Missing price_per_unit:")
print(invalid_ppu[["unique_id","product_name","producthierarchy_level2","price_per_unit"]])

KeyError: 'producthierarchy_level1'

In [39]:
date_columns = [
    "promotion_valid_upto",
    "promotion_valid_from",
    "price_valid_from"
]

for col in date_columns:
    print("\nColumn:", col)
    print(df[col].value_counts(dropna=False))


Column: promotion_valid_upto
promotion_valid_upto
NaN          3366
19.8.2026     131
26.8.2026      87
15.8.2026      20
22.8.2026      11
29.8.2026       4
19.9.2026       1
Name: count, dtype: int64

Column: promotion_valid_from
promotion_valid_from
NaN          3044
13.8.2026     424
20.8.2026      48
24.8.2026      43
17.8.2026      41
10.8.2026      15
6.8.2026        4
15.8.2025       1
Name: count, dtype: int64

Column: price_valid_from
price_valid_from
NaN          3044
13.8.2026     424
20.8.2026      48
24.8.2026      43
17.8.2026      41
10.8.2026      15
6.8.2026        4
15.8.2025       1
Name: count, dtype: int64


In [40]:
invalid_discount = df[
    df["percentage_discount"]
    .astype(str)
    .str.contains(r"[%\-]", regex=True, na=False)
]

print("Invalid percentage_discount values:")
print(invalid_discount[["unique_id", "product_name", "percentage_discount"]])

print("\nCount:", len(invalid_discount))

Invalid percentage_discount values:
Empty DataFrame
Columns: [unique_id, product_name, percentage_discount]
Index: []

Count: 0


In [41]:
s = df['percentage_discount']

mask = (
    s.notna() &
    s.astype(str).str.strip().ne("") &
    ~s.astype(str).str.fullmatch(r'\d+', na=False)
)

issues = df.loc[
    mask,
    ['pdp_url', 'percentage_discount']
].copy()

issues['row_no'] = issues.index + 2

issues = issues[['row_no', 'pdp_url', 'percentage_discount']]

print(issues.to_string(index=False))
print(f"\nTotal issues: {len(issues)}")

 row_no                                                                                                                               pdp_url  percentage_discount
      7                         https://sortiment.lidl.ch/de/catalog/product/view/id/14079/s/montepulciano-d-abruzzo-doc-0151217/category/54/                 20.0
     20                         https://sortiment.lidl.ch/de/catalog/product/view/id/14499/s/prosecco-valdobbiadene-docg-0149516/category/54/                 20.0
     32                                     https://sortiment.lidl.ch/de/catalog/product/view/id/13318/s/buendnerfleisch-5107879/category/54/                 20.0
     81                      https://sortiment.lidl.ch/de/catalog/product/view/id/16188/s/terra-natura-baguette-rustique-5111837/category/54/                 16.0
     93                                               https://sortiment.lidl.ch/de/catalog/product/view/id/21793/s/gyoza-0145799/category/54/                 20.0
    143               

In [51]:
invalid = df[
    df["promotion_description"].notna() &
    (
        df["promotion_valid_from"].isna() |
        df["promotion_valid_upto"].isna()
    )
]

print(f"Rows with promotion description but missing promotion dates: {len(invalid)}")

display(
    invalid[
        [
            "unique_id",
            "product_name",
            "promotion_description",
            "promotion_valid_from",
            "promotion_valid_upto",
            "pdp_url"
        ]
    ]
)

Rows with promotion description but missing promotion dates: 519


,unique_id,product_name,promotion_description,promotion_valid_from,promotion_valid_upto,pdp_url
5,151217,Montepulciano d'Abruzzo DOC,-20%,13.8.2026,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/14079/s/montepulciano-d-abruzzo-doc-0151217/category/54/
18,149516,Prosecco Valdobbiadene DOCG,-20%,13.8.2026,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/14499/s/prosecco-valdobbiadene-docg-0149516/category/54/
30,5107879,Bündnerfleisch,-20%,13.8.2026,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/13318/s/buendnerfleisch-5107879/category/54/
34,5114255,Chicken Wings nature,Aktion,NaN,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21941/s/chicken-wings-nature-5114255/category/54/
35,1801686,Frisco Extrême Cornets Soft Core,Aktion,NaN,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21940/s/frisco-extreme-cornets-soft-core-1801686/category/54/
38,1009870,Glace Erdbeer Crisp XXL,Aktion,NaN,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21939/s/glace-erdbeer-crisp-xxl-1009870/category/54/
42,1021254,Lager Bier XXL,Aktion,NaN,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21938/s/lager-bier-xxl-1021254/category/54/
43,1017455,Babyfeuchttücher 99% Wasser XXL,Aktion,NaN,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21937/s/babyfeuchttuecher-99-wasser-xxl-1017455/category/54/
44,153647,Mini Mix Mandel Vollmilch,Aktion,NaN,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21936/s/mini-mix-mandel-vollmilch-0153647/category/54/
45,218173,Hummus XXL,Aktion,NaN,NaN,https://sortiment.lidl.ch/de/catalog/product/view/id/21934/s/hummus-xxl-0218173/category/54/


In [50]:
import pandas as pd
import numpy as np
import re

issues = []

valid_units = {
    "g","kg","ml","l","stuck","stk","btl","wg","pcs","pc","piece","ea"
}

for i, row in df.iterrows():

    url = str(row.get("pdp_url",""))

    # unique_id
    if pd.isna(row["unique_id"]) or str(row["unique_id"]).strip()=="":
        issues.append([i+1,url,"unique_id","Missing unique_id"])

    # product_name
    if pd.isna(row["product_name"]) or str(row["product_name"]).strip()=="":
        issues.append([i+1,url,"product_name","Missing product_name"])

    # selling_price
    if pd.isna(row["selling_price"]):
        issues.append([i+1,url,"selling_price","Missing selling_price"])

    # regular price < selling price
    if pd.notna(row["regular_price"]) and pd.notna(row["selling_price"]):
        if float(row["selling_price"]) > float(row["regular_price"]):
            issues.append([i+1,url,"price","selling_price greater than regular_price"])

    # promotion price
    if pd.notna(row["promotion_price"]) and pd.notna(row["selling_price"]):
        if float(row["promotion_price"]) > float(row["selling_price"]):
            issues.append([i+1,url,"promotion_price","promotion_price greater than selling_price"])

    # promotion description
    if pd.notna(row["promotion_price"]) and (
        pd.isna(row["promotion_description"]) or
        str(row["promotion_description"]).strip()==""
    ):
        issues.append([i+1,url,"promotion_description",
                       "promotion_price present but promotion_description empty"])

    # grammage quantity
    if pd.isna(row["grammage_quantity"]):
        issues.append([i+1,url,"grammage_quantity","Missing grammage_quantity"])

    # grammage unit
    if pd.isna(row["grammage_unit"]) or str(row["grammage_unit"]).strip()=="":
        issues.append([i+1,url,"grammage_unit","Missing grammage_unit"])
    elif str(row["grammage_unit"]).lower() not in valid_units:
        issues.append([i+1,url,"grammage_unit","Unknown grammage_unit"])

    # price_per_unit
    if pd.isna(row["price_per_unit"]) or str(row["price_per_unit"]).strip()=="":
        issues.append([i+1,url,"price_per_unit","Missing price_per_unit"])

    # currency
    if str(row["currency"]).strip().upper()!="EUR":
        issues.append([i+1,url,"currency","Unexpected currency"])

    # breadcrumb
    if pd.isna(row["breadcrumb"]) or str(row["breadcrumb"]).strip()=="":
        issues.append([i+1,url,"breadcrumb","Empty breadcrumb"])

    # pack_size
    if pd.isna(row["pack_size"]) or str(row["pack_size"]).strip()=="":
        issues.append([i+1,url,"pack_size","Empty pack_size"])

    # site_shown_uom
    if pd.isna(row["site_shown_uom"]) or str(row["site_shown_uom"]).strip()=="":
        issues.append([i+1,url,"site_shown_uom","Empty site_shown_uom"])

    # URL validation
    if not re.match(r"^https?://", url):
        issues.append([i+1,url,"pdp_url","Invalid URL"])

# duplicate unique_id
dup = df[df["unique_id"].duplicated(keep=False)]

for i,row in dup.iterrows():
    issues.append([
        i+1,
        row["pdp_url"],
        "unique_id",
        "Duplicate unique_id"
    ])

issues_df = pd.DataFrame(
    issues,
    columns=["row_no","pdp_url","column","issue"]
)

print("Total Issues:", len(issues_df))
display(issues_df)

Total Issues: 8818


,row_no,pdp_url,column,issue
0,1,https://sortiment.lidl.ch/de/catalog/product/view/id/13737/s/schweizer-aepfel-rot-0080220/category/54/,grammage_unit,Unknown grammage_unit
1,1,https://sortiment.lidl.ch/de/catalog/product/view/id/13737/s/schweizer-aepfel-rot-0080220/category/54/,currency,Unexpected currency
2,1,https://sortiment.lidl.ch/de/catalog/product/view/id/13737/s/schweizer-aepfel-rot-0080220/category/54/,pack_size,Empty pack_size
3,2,https://sortiment.lidl.ch/de/catalog/product/view/id/11808/s/schweizer-zucchetti-0082345/category/54/,grammage_unit,Unknown grammage_unit
4,2,https://sortiment.lidl.ch/de/catalog/product/view/id/11808/s/schweizer-zucchetti-0082345/category/54/,currency,Unexpected currency
5,2,https://sortiment.lidl.ch/de/catalog/product/view/id/11808/s/schweizer-zucchetti-0082345/category/54/,pack_size,Empty pack_size
6,3,https://sortiment.lidl.ch/de/catalog/product/view/id/15344/s/schweizer-rueebli-0082761/category/54/,currency,Unexpected currency
7,3,https://sortiment.lidl.ch/de/catalog/product/view/id/15344/s/schweizer-rueebli-0082761/category/54/,pack_size,Empty pack_size
8,4,https://sortiment.lidl.ch/de/catalog/product/view/id/12790/s/trauben-hell-kernlos-0080505/category/54/,currency,Unexpected currency
9,4,https://sortiment.lidl.ch/de/catalog/product/view/id/12790/s/trauben-hell-kernlos-0080505/category/54/,pack_size,Empty pack_size


In [54]:
mask = (
    df["regular_price"].notna() &
    df["price_was"].notna() &
    (pd.to_numeric(df["regular_price"], errors="coerce") !=
     pd.to_numeric(df["price_was"], errors="coerce"))
)

issues = df.loc[mask, ["pdp_url", "regular_price", "price_was"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "regular_price", "price_was"]]

issues

KeyError: 'price_was'

In [55]:
mask = (
    df["unique_id"].astype(str).fillna("") + "P"
    !=
    df["product_unique_key"].astype(str).fillna("")
)

issues = df.loc[mask, ["unique_id", "product_unique_key", "pdp_url"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "unique_id", "product_unique_key"]]

issues

KeyError: 'product_unique_key'

In [56]:
import pandas as pd

# Normalize for case-insensitive comparison
df["site_shown_uom"] = df["site_shown_uom"].fillna("").astype(str)
df["grammage_unit"] = df["grammage_unit"].fillna("").astype(str)
df["grammage_quantity"] = df["grammage_quantity"].fillna("").astype(str)

# -----------------------------
# Rule 1: Teebeutel / Beutel -> btl
# -----------------------------
btl_keywords = [
    "Teebeutel Packung",
    "Teebeutel",
    "Beutel",
    "Btl",
    "Teebeutel Karton",
    "Teebeutel Paket",
    "Packung Beutel"
]

btl_invalid = df[
    df["site_shown_uom"].str.contains("|".join(btl_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "btl")
]

print(f"BTL unit mismatches: {len(btl_invalid)}")
display(btl_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit"
]])

# -----------------------------
# Rule 2: Portion Packung / ANW -> stuck
# -----------------------------
stuck_keywords = [
    "Portion Packung",
    "ANW"
]

stuck_invalid = df[
    df["site_shown_uom"].str.contains("|".join(stuck_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "stuck")
]

print(f"STUCK unit mismatches: {len(stuck_invalid)}")
display(stuck_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit"
]])

# -----------------------------
# Rule 3: wg / Waschgänge -> wg
# -----------------------------
wg_keywords = [
    "wg",
    "Waschgänge"
]

wg_invalid = df[
    df["site_shown_uom"].str.contains("|".join(wg_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "wg")
]

print(f"WG unit mismatches: {len(wg_invalid)}")
display(wg_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit",
    "pdp_url"
]])

BTL unit mismatches: 160


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit
112,2020003506550,SPAR Müllbeutel mit Duft 30l 20 Stk.,SPAR Müllbeutel mit Duft 30l 20 Stk.,30,l
117,2020003507151,SPAR Öko Müllbeutel 150l 8 Stk.,SPAR Öko Müllbeutel 150l 8 Stk.,150,l
119,2020003471797,SPAR Biomüllbeutel 120l 5 Stk.,SPAR Biomüllbeutel 120l 5 Stk.,120,l
217,2020003471179,SPAR Biomüllbeutel 30l 8 Stk.,SPAR Biomüllbeutel 30l 8 Stk.,30,l
230,2020004008084,SPAR Gefrierbeutel 6l,SPAR Gefrierbeutel 6l,6,l
262,2020004008060,SPAR Gefrierbeutel 1l,SPAR Gefrierbeutel 1l,1,l
263,2020005369887,SPAR Müllbeutel Fixierband,SPAR Müllbeutel Fixierband,1,stück
266,2020006225847,SPAR Müllbeutel Tragegriff 50L,SPAR Müllbeutel Tragegriff 50L,50,l
272,2020003506536,SPAR Müllbeutel mit Duft 20l 20 Stk.,SPAR Müllbeutel mit Duft 20l 20 Stk.,20,l
284,2020006225908,SPAR Müllbeutel Recycling mit Tragegriff 20L,SPAR Müllbeutel Recycling mit Tragegriff 20L,20,l


STUCK unit mismatches: 1


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit
15134,2020002299026,Syoss Cool Blonds Haarcoloration 10-55 Platinum Blond 1 Anwendung,Syoss Cool Blonds Haarcoloration 10-55 Platinum Blond 1 Anwendung,1,stück


WG unit mismatches: 10


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,pdp_url
7461,2020005400351,Siemens Waschmaschine WG44G2Z22 9kg,Siemens Waschmaschine WG44G2Z22 9kg,9,kg,https://www.spar.at/produktwelt/siemens-waschmaschine-9kg-p2020005400351
14399,2020006269643,Gorenje Waschmaschine WG2PS74AP2/AT,Gorenje Waschmaschine WG2PS74AP2/AT,1,stück,https://www.spar.at/produktwelt/gorenje-waschmaschine-wg2ps74ap2-at-p2020006269643
24253,2020005407312,Bosch Waschmaschine WGE0241H 7 kg,Bosch Waschmaschine WGE0241H 7 kg,7,kg,https://www.spar.at/produktwelt/bosch-waschmaschine-wge0241h-7-kg-p2020005407312
30356,2020006269452,Gorenje Waschmaschine WG474A2P1,Gorenje Waschmaschine WG474A2P1,1,stück,https://www.spar.at/produktwelt/gorenje-waschmaschine-wg474a2p1-p2020006269452
31588,2020005906921,Siemens Waschmaschine iQ500 WG44G2ZEM,Siemens Waschmaschine iQ500 WG44G2ZEM,1,stück,https://www.spar.at/produktwelt/siemens-waschmaschine-iq500-wg44g2zem-p2020005906921
33141,2020005032477,Gorenje Waschmaschine WGPNEI14A2DTS 10 kg,Gorenje Waschmaschine WGPNEI14A2DTS 10 kg,10,kg,https://www.spar.at/produktwelt/gorenje-waschmaschine-wgpnei14a2dts-10-kg-p2020005032477
35395,2020006269674,Gorenje Waschmaschine WG2PS84A3P2/AT,Gorenje Waschmaschine WG2PS84A3P2/AT,1,stück,https://www.spar.at/produktwelt/gorenje-waschmaschine-wg2ps84a3p2-at-p2020006269674
35661,2020005680852,Siemens Waschmaschine WG46B2071 iQ700 9 kg,Siemens Waschmaschine WG46B2071 iQ700 9 kg,9,kg,https://www.spar.at/produktwelt/siemens-waschmaschine-wg46b2071-iq700-p2020005680852
35950,2020005677111,Siemens Waschmaschine WG44G2F 9 kg,Siemens Waschmaschine WG44G2F 9 kg,9,kg,https://www.spar.at/produktwelt/siemens-waschmaschine-wg44g2f-p2020005677111
38770,2020004680020,Siemens Waschmaschine WG44B20X40 9 kg,Siemens Waschmaschine WG44B20X40 9 kg,9,kg,https://www.spar.at/produktwelt/siemens-waschmaschine-wg44b20x40-p2020004680020


In [56]:
import pandas as pd

price_cols = ["regular_price", "selling_price"]

for col in price_cols:
    # Ignore missing values
    mask = (
        df[col].notna() &
        (~df[col].astype(float).round(2).eq(df[col].astype(float)))
    )

    invalid = df.loc[mask, ["unique_id", "product_name", col]]

    print(f"\n{col}: {len(invalid)} values not in 2-decimal format")
    display(invalid)


regular_price: 0 values not in 2-decimal format


,unique_id,product_name,regular_price



selling_price: 0 values not in 2-decimal format


,unique_id,product_name,selling_price


In [57]:
import pandas as pd

# Match both old and new site keywords
promo_mask = (
    df["promotion_description"]
    .fillna("")
    .astype(str)
    .str.contains(r"Trajno znižano|Znižano", case=False, na=False)
)

# Empty promotion price
promo_price_empty = (
    df["promotion_price"].isna()
    | df["promotion_price"].astype(str).str.strip().eq("")
)

# Check where promotion price is NOT empty
promo_price_issue = promo_mask & ~promo_price_empty

# Get all matching records
issues = df.loc[
    promo_mask,
    [
        "unique_id",
        "pdp_url",
        "promotion_description",
        "regular_price",
        "selling_price",
        "promotion_price"
    ]
].copy()

issues["row_no"] = issues.index + 2

# Add issue remarks
def check_issue(row):
    remarks = []

    if pd.notna(row["promotion_price"]) and str(row["promotion_price"]).strip() != "":
        remarks.append("promotion_price should be empty")

    if row["regular_price"] != row["selling_price"]:
        remarks.append("regular_price and selling_price are not equal")

    return "; ".join(remarks)

issues["issue"] = issues.apply(check_issue, axis=1)

# Show only violations
issues = issues[issues["issue"] != ""]

issues = issues[
    [
        "row_no",
        "unique_id",
        "pdp_url",
        "promotion_description",
        "regular_price",
        "selling_price",
        "promotion_price",
        "issue"
    ]
]

print(issues.to_string(index=False))

Empty DataFrame
Columns: [row_no, unique_id, pdp_url, promotion_description, regular_price, selling_price, promotion_price, issue]
Index: []


In [58]:
import re
import pandas as pd

# Normalize grammage_unit
df["grammage_unit_norm"] = df["grammage_unit"].fillna("").str.lower().str.strip()

# Extract quantity from site_shown_uom
df["site_qty"] = (
    df["site_shown_uom"]
      .str.extract(r"(\d+(?:[.,]\d+)?)", expand=False)
      .str.replace(",", ".", regex=False)
)

df["site_qty"] = pd.to_numeric(df["site_qty"], errors="coerce")

# ---------- Rule 1 : Beutel / Btl ----------
btl_pattern = (
    r"Teebeutel Packung|Teebeutel Karton|Teebeutel Paket|"
    r"Packung Beutel|Teebeutel|Beutel|Btl"
)

btl_issue = df[
    df["site_shown_uom"].fillna("").str.contains(btl_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "btl") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

# ---------- Rule 2 : Portion Packung / ANW ----------
stuck_pattern = r"Portion Packung|ANW"

stuck_issue = df[
    df["site_shown_uom"].fillna("").str.contains(stuck_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "stuck") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

# ---------- Rule 3 : Waschgang / Waschgänge / wg ----------
wg_pattern = r"\bwg\b|Waschgänge|Waschgang"

wg_issue = df[
    df["site_shown_uom"].fillna("").str.contains(wg_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "wg") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

print("Btl issues:", len(btl_issue))
print("Stuck issues:", len(stuck_issue))
print("WG issues:", len(wg_issue))

Btl issues: 1162
Stuck issues: 0
WG issues: 83


In [59]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

results = []

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r" {2,}", regex=True)

    for idx, value in df.loc[mask, col].items():
        results.append({
            "row_no": idx + 2,
            "pdp_url": df.at[idx, "pdp_url"],
            "column": col,
            "issue_value": repr(value)
        })

issues_df = pd.DataFrame(results)
issues_df

""


In [60]:
btl_issue[
    [
        "unique_id",
        "product_name",
        "site_shown_uom",
        "grammage_quantity",
        "grammage_unit",
        "site_qty"
    ]
].head(20)

,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,site_qty
11,00-101513,Toppits Gefrierbeutel 6 Liter 20 Beutel Packung,20 Beutel Packung,1,stück,20.0
12,00-101511,Toppits Gefrierbeutel 1L 40 Beutel Packung,40 Beutel Packung,1,stück,40.0
48,00-116129,Kinder Schokobons 200 g Beutel,200 g Beutel,200,g,200.0
62,00-103992,Vegeta Würzmischung 1 kg Beutel,1 kg Beutel,1,kg,1.0
68,00-11697,Teekanne Früchtetee 20 Teebeutel Packung,20 Teebeutel Packung,20,btl,20.0
92,00-11080,Knorr Basis für Faschiertes 1 Packung Beutel,1 Packung Beutel,1,btl,1.0
98,00-129148,Teekanne Fixbutte 40 Teebeutel Paket,40 Teebeutel Paket,40,btl,40.0
99,00-129163,Teekanne Fixminze 40 Teebeutel Paket,40 Teebeutel Paket,40,btl,40.0
100,00-129189,Teekanne Fixmille 40 Teebeutel Packung,40 Teebeutel Packung,40,btl,40.0
104,00-133728,Twinings English Breakfast 50 Teebeutel Packung,50 Teebeutel Packung,50,btl,50.0


In [61]:
wrong_unit_wg = wg_issue[
    wg_issue["grammage_unit_norm"] != "wg"
]

wrong_qty_wg = wg_issue[
    (wg_issue["grammage_unit_norm"] == "wg") &
    (wg_issue["grammage_quantity"] != wg_issue["site_qty"])
]

print("WG - Wrong unit:", len(wrong_unit_wg))
print("WG - Wrong quantity:", len(wrong_qty_wg))

WG - Wrong unit: 79
WG - Wrong quantity: 4


In [62]:
wrong_unit_wg[
    [
        "unique_id",
        "product_name",
        "site_shown_uom",
        "grammage_quantity",
        "grammage_unit",
        "pdp_url"
    ]
]

,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,pdp_url
2096,00-393820,Persil Gel Color Activ 100 Waschgang Flasche,100 Waschgang Flasche,1,stück,https://shop.Billa.at/produkte/persil-gel-color-activ-00393820
3530,00-451977,bi good Lavendel Vollwaschmittel Beutel 25 Waschgang Beutel,25 Waschgang Beutel,1,stück,https://shop.Billa.at/produkte/bi-good-lavendel-vollwaschmittel-beutel-00451977
3533,00-451972,bi good Apfelblüte Colorwaschmittel Beutel 25 Waschgang Beutel,25 Waschgang Beutel,1,stück,https://shop.Billa.at/produkte/bi-good-apfelbluete-colorwaschmittel-beutel-00451972
3655,00-465128,Sagrotan Hygienespüler 20 Waschgang Flasche,20 Waschgang Flasche,1,stück,https://shop.Billa.at/produkte/sagrotan-hygienespueler-00465128
3814,00-471467,Lenor Weichspüler Goldene Orchidee 38 Waschgang Flasche,38 Waschgang Flasche,1,stück,https://shop.Billa.at/produkte/lenor-weichspueler-goldene-orchidee-00471467
4021,00-482549,Weißer Riese Trio Caps Orchidee 40 Waschgang Karton,40 Waschgang Karton,1,stück,https://shop.Billa.at/produkte/weisser-riese-trio-caps-orchidee-00482549
4022,00-482548,Weißer Riese Trio Caps Lotus 40 Waschgang Karton,40 Waschgang Karton,1,stück,https://shop.Billa.at/produkte/weisser-riese-trio-caps-lotus-00482548
4848,00-572851,Dr. Beckmann Magic Leaves Universal 25 Waschgang Stück,25 Waschgang Stück,1,stück,https://shop.Billa.at/produkte/dr-beckmann-magic-leaves-universal-00572851
4885,00-575505,Clever Weichspüler Pfirsich 60 Waschgang Flasche,60 Waschgang Flasche,1,stück,https://shop.Billa.at/produkte/clever-weichspueler-pfirsich-00575505
4890,00-575545,Clever Weichspüler Frische Quelle 60 Waschgang Flasche,60 Waschgang Flasche,1,stück,https://shop.Billa.at/produkte/clever-weichspueler-frische-quelle-00575545


In [63]:
# JO PREIS price structure
jo_mask = df['promotion_description'].fillna('').astype(str).str.contains(
    r'JO PREIS',
    case=False,
    na=False
)

issues = df.loc[
    jo_mask & (
        df['regular_price'].isna() |
        df['selling_price'].isna() |
        df['price_per_unit'].isna()
    ),
    [
        'pdp_url',
        'promotion_description',
        'regular_price',
        'selling_price',
        'price_per_unit'
    ]
].copy()

issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'price_per_unit']
]

print(f"JO PREIS issues: {len(issues)}")
print(issues.to_string(index=False))

JO PREIS issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, regular_price, selling_price, price_per_unit]
Index: []


In [64]:
text_cols = df.select_dtypes(include="object").columns

leading_trailing = {}

for col in text_cols:
    mask = (
        df[col].fillna("").astype(str)
        != df[col].fillna("").astype(str).str.strip()
    )
    if mask.any():
        leading_trailing[col] = df.loc[mask, col]

print(leading_trailing.keys())

dict_keys(['organictype'])


In [65]:
multiple_spaces = {}

for col in text_cols:
    mask = df[col].fillna("").str.contains(r"\s{2,}", regex=True)
    if mask.any():
        multiple_spaces[col] = df.loc[mask, col]

print(multiple_spaces.keys())

dict_keys([])


In [66]:
special_space = {}

for col in text_cols:
    mask = df[col].fillna("").str.contains(r"[\t\r\n]", regex=True)
    if mask.any():
        special_space[col] = df.loc[mask, col]

print(special_space.keys())

dict_keys([])


In [67]:
special_space

{}

In [6]:
issues = []

for col in text_cols:
    mask = (
        (df[col].fillna("") != df[col].fillna("").str.strip()) |
        df[col].fillna("").str.contains(r"\s{2,}", regex=True) |
        df[col].fillna("").str.contains(r"[\t\r\n]", regex=True) |
        df[col].fillna("").str.contains("\u00A0", regex=False)
    )

    if mask.any():
        tmp = df.loc[mask, ["unique_id", col]].copy()
        tmp["column"] = col
        issues.append(tmp)

issues = pd.concat(issues, ignore_index=True)

issues.head()

NameError: name 'text_cols' is not defined

In [49]:
import pandas as pd

# Convert to datetime for proper date comparison
price_date = pd.to_datetime(df["price_valid_from"], errors="coerce")
promo_date = pd.to_datetime(df["promotion_valid_from"], errors="coerce")

# Find rows where both have values but are different
mask = (
    price_date.notna()
    & promo_date.notna()
    & (price_date != promo_date)
)

issues = df.loc[
    mask,
    ["unique_id", "pdp_url", "price_valid_from", "promotion_valid_from"]
].copy()

issues["row_no"] = issues.index + 2

issues = issues[
    ["row_no", "unique_id", "pdp_url",
     "price_valid_from", "promotion_valid_from"]
]

print(issues.to_string(index=False))
print(f"\nTotal issues: {len(issues)}")

Empty DataFrame
Columns: [row_no, unique_id, pdp_url, price_valid_from, promotion_valid_from]
Index: []

Total issues: 0


/tmp/ipykernel_24657/158468668.py:4: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  price_date = pd.to_datetime(df["price_valid_from"], errors="coerce")
/tmp/ipykernel_24657/158468668.py:5: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  promo_date = pd.to_datetime(df["promotion_valid_from"], errors="coerce")


In [8]:
import pandas as pd

# Convert price columns to numeric
cols = ["regular_price", "selling_price", "promotion_price", "price_per_unit"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

invalid = df[
    # Mehrweg product
    (
        df["product_name"].fillna("").str.contains("Mehrweg", case=False) |
        df["breadcrumb"].fillna("").str.contains("Mehrweg", case=False)
    )
    &
    # Has AKTION promotion
    df["promotion_description"].fillna("").str.contains("AKTION", case=False)
    &
    # Rule violation
    (
        df["regular_price"].notna() |
        (df["selling_price"] != df["price_per_unit"]) |
        (df["promotion_price"] != df["price_per_unit"])
    )
]

print("Invalid rows:", len(invalid))

invalid[
    [
        "unique_id",
        "product_name",
        "regular_price",
        "selling_price",
        "promotion_price",
        "price_per_unit",
        "promotion_description",
        "pdp_url",
    ]
]

Invalid rows: 0


,unique_id,product_name,regular_price,selling_price,promotion_price,price_per_unit,promotion_description,pdp_url


In [9]:
import pandas as pd

# 1. Identify which IDs are duplicated
duplicate_ids = df[df.duplicated(subset=['unique_id'], keep=False)]

# 2. Count occurrences of each duplicate ID
duplicate_counts = df['unique_id'].value_counts()
duplicate_counts = duplicate_counts[duplicate_counts > 1]


In [10]:
duplicate_ids

,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom


In [11]:
df[df.regular_price.isna()]

,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom
322,00-21841,billa,2026-08-14,Gösser Märzen 500 ml,500,ml,NaN,1.06,NaN,NaN,NaN,1.06,NaN,NaN,AKTION,EUR,Alle Kategorien > Getränke > Bier > Flaschenbier,https://shop.billa.at/produkte/goesser-maerzen-0021841,NaN,NaN,500 ml
331,00-21982,billa,2026-08-14,Stiegl Goldbräu 500 ml,500,ml,NaN,0.99,NaN,NaN,NaN,0.99,NaN,NaN,AKTION,EUR,Alle Kategorien > Getränke > Bier > Flaschenbier,https://shop.billa.at/produkte/stiegl-goldbraeu-0021982,NaN,NaN,500 ml
1046,00-30036,billa,2026-08-14,Gösser Märzen 6er 500 ml,500,ml,NaN,1.06,NaN,NaN,NaN,1.06,NaN,NaN,AKTION,EUR,Alle Kategorien > Getränke > Bier > Flaschenbier,https://shop.billa.at/produkte/goesser-maerzen-6er-0030036,NaN,NaN,500 ml
1281,00-317777,billa,2026-08-14,Alnatura Tomatensauce Basilikum 350 g Glas,350,g,NaN,1.49,NaN,NaN,NaN,1.49,NaN,NaN,AKTION,EUR,"Alle Kategorien > Vorratsschrank > Tomatenprodukte, Sugo & Pesto > Sugo",https://shop.billa.at/produkte/alnatura-tomatensauce-basilikum-00317777,NaN,NaN,350 g Glas
1803,00-377843,billa,2026-08-14,Stiegl Hell 500 ml,500,ml,NaN,0.99,NaN,NaN,NaN,0.99,NaN,NaN,AKTION,EUR,Alle Kategorien > Getränke > Bier > Flaschenbier,https://shop.billa.at/produkte/stiegl-hell-00377843,NaN,NaN,500 ml
1908,00-381753,billa,2026-08-14,Lindt Lindor Kugeln Dark 500 g Packung,500,g,NaN,9.99,NaN,NaN,NaN,9.99,NaN,NaN,AKTION,EUR,Alle Kategorien > Vorratsschrank > Süßes & Salziges > Süßes > Süße Spezialitäten,https://shop.billa.at/produkte/lindt-lindor-kugeln-dark-00381753,NaN,NaN,500 g Packung
2525,00-411889,billa,2026-08-14,Stiegl Goldbräu 6er 500 ml,500,ml,NaN,0.99,NaN,NaN,NaN,0.99,NaN,NaN,AKTION,EUR,Alle Kategorien > Getränke > Bier > Flaschenbier,https://shop.billa.at/produkte/stiegl-goldbraeu-6er-00411889,NaN,NaN,500 ml
4167,00-504120,billa,2026-08-14,"Stiegl Paracelsus Bio Zwickl 6x0,5l 500 ml",500,ml,NaN,1.19,NaN,NaN,NaN,1.19,NaN,NaN,AKTION,EUR,Alle Kategorien > Getränke > Bier > Flaschenbier,https://shop.billa.at/produkte/stiegl-paracelsus-bio-zwickl-6x05l-00504120,NaN,NaN,500 ml
4715,00-573700,billa,2026-08-14,BILLA Apfelsaft 1 Liter,1,l,NaN,1.59,NaN,NaN,NaN,1.59,NaN,NaN,AKTION,EUR,Alle Kategorien > Getränke > Alkoholfreie Getränke > Frucht- & Gemüsesäfte,https://shop.billa.at/produkte/billa-apfelsaft-00573700,NaN,NaN,1 Liter
5729,00-678591,billa,2026-08-14,Alnatura Tomate Kräutersauce 350 g Glas,350,g,NaN,1.49,NaN,NaN,NaN,1.49,NaN,NaN,AKTION,EUR,Alle Kategorien > Rein Pflanzlich > Vorratsschrank > Tomatenprodukte & Pesto,https://shop.billa.at/produkte/alnatura-tomate-kraeutersauce-00678591,NaN,NaN,350 g Glas


In [12]:
df.grammage_unit.value_counts()

grammage_unit
g        7252
l        2079
ml       1179
stück    1158
kg        410
btl       220
wg          4
Name: count, dtype: int64

In [13]:
df.store_addressline1.value_counts()

AttributeError: 'DataFrame' object has no attribute 'store_addressline1'

In [14]:
duplicates = df[
    df.duplicated(subset=["store_addressline1", "unique_id"], keep=False)
].sort_values(["store_addressline1", "unique_id"])

print("Duplicate unique_id within stores:", len(duplicates))

display(duplicates[["store_addressline1", "unique_id"]])


KeyError: Index(['store_addressline1'], dtype='object')

In [15]:
df.dtypes

unique_id                 object
competitor_name           object
extraction_date           object
product_name              object
grammage_quantity         object
grammage_unit             object
regular_price            float64
selling_price            float64
price_valid_from         float64
price_per_unit           float64
percentage_discount      float64
promotion_price          float64
promotion_valid_from     float64
promotion_valid_upto     float64
promotion_description     object
currency                  object
breadcrumb                object
pdp_url                   object
region                   float64
pack_size                float64
site_shown_uom            object
dtype: object

In [16]:
issues = df[
    df["grammage_quantity"].astype(str).str.count(r"\d+\s*x") > 1
][["grammage_quantity", "pdp_url"]].reset_index(names="row_no")

issues

,row_no,grammage_quantity,pdp_url


In [17]:
wine = df[
    df["producthierarchy_level1"].astype(str).str.contains("wein|wine", case=False, na=False)
]

issues = wine[
    wine["site_shown_uom"].astype(str).str.contains(r"\b[2-9]\d*\s*x\b", regex=True, na=False)
][["site_shown_uom", "regular_price", "selling_price", "pdp_url"]]

issues.reset_index(names="row_no")

KeyError: 'producthierarchy_level1'

In [48]:
import pandas as pd

issues = []

for idx, row in df.iterrows():

    rp = str(row.get("regular_price", "")).strip()
    sp = str(row.get("selling_price", "")).strip()
    pp = str(row.get("promotion_price", "")).strip()
    pdesc = str(row.get("promotion_description", "")).strip()

    # Ignore NaN represented as string
    if rp.lower() == "nan": rp = ""
    if sp.lower() == "nan": sp = ""
    if pp.lower() == "nan": pp = ""
    if pdesc.lower() == "nan": pdesc = ""

    remarks = []

    # Rule 1: Regular price contains comma
    if "," in rp:
        remarks.append("Regular price contains comma (,) instead of dot (.)")

    # Rule 2: Regular price missing
    if rp == "" and sp != "":
        remarks.append("Regular price missing (should be assigned from selling price)")

    # Rule 3: Promotion description missing but prices exist
    if pdesc == "" and rp != "" and sp != "":
        if pp != sp:
            remarks.append("Promotion description missing - promotion_price should equal selling_price")

    # Rule 4: Aktion products with all prices (reference only)
    if "aktion" in pdesc.lower():
        if rp == "":
            remarks.append("Aktion product with missing regular price (requires previous iteration data)")
        elif pp == "":
            remarks.append("Aktion product missing promotion_price")

    if remarks:
        issues.append({
            "row_no": idx + 1,
            "profile_url": row.get("profile_url", ""),
            "regular_price": rp,
            "selling_price": sp,
            "promotion_price": pp,
            "promotion_description": pdesc,
            "issue": " | ".join(remarks)
        })

issues_df = pd.DataFrame(issues)

print(f"Total issues found: {len(issues_df)}")
display(issues_df)

Total issues found: 3272


,row_no,profile_url,regular_price,selling_price,promotion_price,promotion_description,issue
0,1,,2.79,2.79,,,Promotion description missing - promotion_price should equal selling_price
1,2,,2.79,2.79,,,Promotion description missing - promotion_price should equal selling_price
2,3,,1.95,1.95,,,Promotion description missing - promotion_price should equal selling_price
3,4,,1.75,1.75,,,Promotion description missing - promotion_price should equal selling_price
4,5,,2.99,2.99,,,Promotion description missing - promotion_price should equal selling_price
5,7,,3.79,3.79,,,Promotion description missing - promotion_price should equal selling_price
6,8,,3.99,3.99,,,Promotion description missing - promotion_price should equal selling_price
7,9,,3.89,3.89,,,Promotion description missing - promotion_price should equal selling_price
8,10,,2.49,2.49,,,Promotion description missing - promotion_price should equal selling_price
9,11,,2.49,2.49,,,Promotion description missing - promotion_price should equal selling_price


In [19]:
df.unique_id.duplicated().sum()

np.int64(0)

In [47]:
mask = (
    df['selling_price'].eq(df['regular_price']) &
    df['selling_price'].eq(df['promotion_price'])
)

issues = df.loc[mask, ['pdp_url', 'selling_price', 'regular_price', "promotion_description", 'promotion_price']].copy()
issues['row_no'] = issues.index + 2

issues[['row_no', 'pdp_url', 'selling_price', 'regular_price', "promotion_description", 'promotion_price']]

,row_no,pdp_url,selling_price,regular_price,promotion_description,promotion_price


In [46]:
# Find Mehrweg products that are not yogurt
# and have missing price_per_unit

mehrweg_mask = (
    df['promotion_description']
    .fillna('')
    .astype(str)
    .str.contains('Mehrweg', case=False, na=False)
)

# Exclude yogurt products
not_yogurt_mask = ~df['product_name'].fillna('').astype(str).str.contains(
    'yogurt|joghurt|yoghurt',
    case=False,
    na=False
)

# price_per_unit is missing / empty
price_per_unit_missing = (
    df['price_per_unit'].isna() |
    df['price_per_unit'].astype(str).str.strip().eq('')
)

mask = mehrweg_mask & not_yogurt_mask & price_per_unit_missing

issues = df.loc[
    mask,
    ['pdp_url', 'product_name', 'promotion_description', 'price_per_unit']
].copy()

# Excel row number
issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'product_name',
     'promotion_description', 'price_per_unit']
]

print(f"Total Mehrweg issues: {len(issues)}")
print(issues.to_string(index=False))

Total Mehrweg issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, product_name, promotion_description, price_per_unit]
Index: []


In [45]:
# Check if selling_price or promotion_price is greater than regular_price

regular_price = pd.to_numeric(df['regular_price'], errors='coerce')
selling_price = pd.to_numeric(df['selling_price'], errors='coerce')
promotion_price = pd.to_numeric(df['promotion_price'], errors='coerce')

mask = (
    regular_price.notna() &
    (
        (selling_price.notna() & (selling_price > regular_price)) |
        (promotion_price.notna() & (promotion_price > regular_price))
    )
)

issues = df.loc[
    mask,
    ['pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'promotion_price']
].copy()

# Excel row number
issues['row_no'] = issues.index + 2

issues = issues[
    ['row_no', 'pdp_url', 'promotion_description',
     'regular_price', 'selling_price', 'promotion_price']
]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, promotion_description, regular_price, selling_price, promotion_price]
Index: []


In [44]:
mask = (
    df["regular_price"].notna() &
    df["price_was"].notna() &
    (pd.to_numeric(df["regular_price"], errors="coerce") ==
     pd.to_numeric(df["price_was"], errors="coerce"))
)

issues = df.loc[mask, ["pdp_url", "regular_price", "price_was"]].copy()
issues["row_no"] = issues.index + 2

issues = issues[["row_no", "pdp_url", "regular_price", "price_was"]]

issues

KeyError: 'price_was'

In [61]:
df.organictype.value_counts()

organictype
Non-Organic    15183
Organic         1630
Name: count, dtype: int64

In [60]:
for i, ch in enumerate(url):
    if ch == " ":
        print(i, "Normal space")
    elif ch == "\xa0":
        print(i, "NBSP")

NameError: name 'url' is not defined

In [26]:
issues = df[
    df["product_unique_key"].astype(str) != (df["unique_id"].astype(str) + "P")
][["unique_id", "product_unique_key", "pdp_url"]].reset_index(names="row_no")

issues


KeyError: 'product_unique_key'

In [27]:
invalid_urls = [u.replace(" ", "\xa0") for u in invalid_urls]

issues = df[df["pdp_url"].isin(invalid_urls)][
    ["unique_id", "pdp_url"]
].reset_index(names="row_no")

issues

NameError: name 'invalid_urls' is not defined

In [28]:
df[df['site_shown_uom'] == ""]


,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom


In [29]:
df.dtypes


unique_id                 object
competitor_name           object
extraction_date           object
product_name              object
grammage_quantity         object
grammage_unit             object
regular_price            float64
selling_price            float64
price_valid_from         float64
price_per_unit           float64
percentage_discount      float64
promotion_price          float64
promotion_valid_from     float64
promotion_valid_upto     float64
promotion_description     object
currency                  object
breadcrumb                object
pdp_url                   object
region                   float64
pack_size                float64
site_shown_uom            object
dtype: object

In [30]:
df.grammage_quantity.value_counts()

grammage_quantity
1        1528
500       784
250       754
200       589
100       585
0.75      531
400       493
150       419
300       320
0.5       279
0.7       256
125       215
0.33      172
180       145
330       142
1 000     135
50        134
20        133
1.5       132
350       124
75        118
750       117
450       117
80        107
0.25      106
160       100
190        98
120        95
2          88
175        86
800        86
600        84
360        78
220        76
70         75
60         73
90         73
140        72
40         69
130        62
10         58
340        55
0.2        53
30         52
700        52
240        51
85         50
225        46
110        45
25         42
3          41
280        40
270        40
170        39
320        37
375        37
95         37
230        36
650        35
45         33
210        33
4          31
460        29
185        29
370        28
670        28
18         28
55         27
15         26
550        26
6 

In [31]:
import pandas as pd
import re

issues = []

# Check all text/object columns
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    s = df[col].fillna("").astype(str)

    # 1. Value ending with comma
    mask_comma = s.str.rstrip().str.endswith(",")

    if mask_comma.any():
        temp = df.loc[mask_comma, [col]].copy()
        temp["row_no"] = temp.index + 2
        temp["column"] = col
        temp["issue"] = "Value ends with comma (,)"
        temp["issue_value"] = temp[col]

        # Add PDP URL if available
        if "pdp_url" in df.columns:
            temp["URL"] = df.loc[mask_comma, "pdp_url"]

        issues.append(
            temp[["row_no", "URL", "column", "issue_value", "issue"]]
        )

    # 2. Leading/trailing whitespace
    mask_space = s.ne(s.str.strip()) & s.ne("")

    if mask_space.any():
        temp = df.loc[mask_space, [col]].copy()
        temp["row_no"] = temp.index + 2
        temp["column"] = col
        temp["issue"] = "Leading/trailing whitespace"
        temp["issue_value"] = temp[col]

        if "pdp_url" in df.columns:
            temp["URL"] = df.loc[mask_space, "pdp_url"]

        issues.append(
            temp[["row_no", "URL", "column", "issue_value", "issue"]]
        )

# Combine results
if issues:
    result = pd.concat(issues, ignore_index=True)

    print(result.to_string(index=False))
else:
    print("No issues found.")

No issues found.


In [32]:
price_cols = ['regular_price', 'selling_price']

for col in price_cols:
    # Convert to string and check exactly 2 decimal places
    mask = (
        df[col].notna() &
        ~df[col].astype(str).str.match(r'^\d+\.\d{2}$')
    )

    if mask.any():
        print(f"\n=== {col} - Invalid 2 Decimal Format ===")
        print(df.loc[mask, ['pdp_url', col]].to_string(index=False))
        print(f"Total issues: {mask.sum()}")
    else:
        print(f"{col}: All values are in 2 decimal format.")


=== regular_price - Invalid 2 Decimal Format ===
                                                                                          pdp_url  regular_price
                                     https://shop.billa.at/produkte/hennessy-very-special-0011379           47.9
                        https://shop.billa.at/produkte/rauch-weizenmehl-extra-fuer-kuchen-0015865            1.8
                                        https://shop.billa.at/produkte/oelz-mehrkorntoast-0014336            3.8
                                https://shop.billa.at/produkte/stangl-fadennudeln-mit-ei-00196840            3.8
                                 https://shop.billa.at/produkte/stangl-bandnudeln-mit-ei-00196865            3.8
                                   https://shop.billa.at/produkte/stangl-fleckerl-mit-ei-00196873            3.8
                                   https://shop.billa.at/produkte/stangl-spiralen-mit-ei-00196881            3.8
                                  https://shop

In [43]:
price_cols = ['regular_price', 'selling_price']

for col in price_cols:
    # Check exactly 2 decimal places
    mask = (
        df[col].notna() &
        ~df[col].astype(str).str.match(r'^\d+\.\d{2}$')
    )

    if mask.any():
        issues = df.loc[mask, ['pdp_url', col]].copy()

        # Row number = dataframe index + 2
        issues.insert(0, 'row_no', issues.index + 2)

        print(f"\n=== {col} - Invalid 2 Decimal Format ===")
        print(issues.to_string(index=False))
        print(f"\nTotal issues: {len(issues)}")

    else:
        print(f"{col}: All values are in 2 decimal format.")


=== regular_price - Invalid 2 Decimal Format ===
 row_no                                                                                                            pdp_url  regular_price
    192               https://sortiment.lidl.ch/de/catalog/product/view/id/21483/s/spezialwaschmittel-1012861/category/54/            2.6
    433             https://sortiment.lidl.ch/de/catalog/product/view/id/20849/s/schweizer-kartoffeln-0083292/category/54/            3.3
    645 https://sortiment.lidl.ch/de/catalog/product/view/id/20044/s/calanda-radler-mango-spritz-0-0--5113623/category/54/            8.5
    998               https://sortiment.lidl.ch/de/catalog/product/view/id/18360/s/recybag-sammelsack-5113234/category/54/           16.0
   1003               https://sortiment.lidl.ch/de/catalog/product/view/id/18359/s/recybag-sammelsack-5113221/category/54/           10.0
   1005               https://sortiment.lidl.ch/de/catalog/product/view/id/18358/s/recybag-sammelsack-5113220/category/54/

In [55]:
# Check whether price_was is the same as regular_price
# If they are different, the record is reported as an issue.

mask = (
    df['price_was'].notna() &
    df['regular_price'].notna() &
    (df['price_was'].astype(str) != df['regular_price'].astype(str))
)

issues = df.loc[
    mask,
    ['pdp_url', 'price_was', 'regular_price']
].copy()

# Add Excel row number (+2 because row 1 is the header)
issues['row_no'] = issues.index + 2

# Reorder columns
issues = issues[['row_no', 'pdp_url', 'price_was', 'regular_price']]

print(f"Total issues: {len(issues)}")
print(issues.to_string(index=False))

Total issues: 0
Empty DataFrame
Columns: [row_no, pdp_url, price_was, regular_price]
Index: []
